In [1]:
# !unzip -q -o ./data/sirius-nlp-contest.zip -d ./data/
# !tar -xzvf ./models/baai-transformers-bge-small-en-v1.5-v1.tar.gz -C ./models/

In [2]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

## Dependencies

In [3]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

import torch
import torch.nn.functional as F
import transformers
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoModel

from catboost import CatBoostClassifier

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

import matplotlib.pyplot as plt
import seaborn as sns

import re
import emoji

from tqdm.cli import tqdm
from tqdm import trange

tqdm.pandas()

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('names', quiet=True)

True

## EDA

In [4]:
df_train = pd.read_csv("./data/train.csv")
df_test_doc = pd.read_csv('./data/test-docs.csv')
df_test_questions = pd.read_csv('./data/test-questions.csv')
df_train.head()

,question,context,c_id
0,To whom did the Virgin Mary allegedly appear i...,"Architecturally, the school has a Catholic cha...",0
1,What is in front of the Notre Dame Main Building?,"Architecturally, the school has a Catholic cha...",0
2,The Basilica of the Sacred heart at Notre Dame...,"Architecturally, the school has a Catholic cha...",0
3,What is the Grotto at Notre Dame?,"Architecturally, the school has a Catholic cha...",0
4,What sits on top of the Main Building at Notre...,"Architecturally, the school has a Catholic cha...",0


In [5]:
df_train.shape

(87599, 3)

In [6]:
df_train['question'].nunique()

87355

In [7]:
df_train['context'].nunique()

18891

In [8]:
df_train['c_id'].describe()

count    87599.000000
mean      9477.616000
std       5495.417059
min          0.000000
25%       4903.000000
50%       9458.000000
75%      14245.000000
max      18890.000000
Name: c_id, dtype: float64

In [9]:
df_test_doc.head()

,context
0,China is the country with the largest populati...
1,The bulk of remaining commercial flight offeri...
2,Based on the similar shifts underway the natio...
3,"Since 2004, through the V&A + RIBA Architectur..."
4,"Society in the Japanese ""Tokugawa period"" (Edo..."


In [10]:
train_context = set(df_train['context'].to_list())
test_context = set(df_test_doc['context'].to_list())

len(train_context), len(test_context)

(18891, 18891)

In [11]:
len(train_context & test_context)

18891

In [12]:
id_to_context = df_test_doc['context'].to_dict()
context_to_id = {v: k for k, v in id_to_context.items()}

In [13]:
documents = df_train[['context', 'c_id']].drop_duplicates().reset_index(drop=True)
documents.head()

,context,c_id
0,"Architecturally, the school has a Catholic cha...",0
1,"As at most other universities, Notre Dame's st...",1
2,The university is the major seat of the Congre...,2
3,The College of Engineering was established in ...,3
4,All of Notre Dame's undergraduate students are...,4


## Text preparing

In [14]:
def text_normalize(text: str) -> str:
    text = re.sub(r'https?://\S+|www\.\S+', ' __URL__ ', text)
    text = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', ' __EMAIL__ ', text)
    text = re.sub(r'\b\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}\b', ' __IP__ ', text)
    text = re.sub(r'[\+]?[(]?[0-9]{1,4}[)]?[-\s\./0-9]{7,}', ' __PHONE__ ', text)
    text = re.sub(r'\b\d{1,2}[/\-\.]\d{1,2}[/\-\.]\d{2,4}\b', ' __DATE__ ', text)
    text = re.sub(r'\b\d{1,2}:\d{2}(?::\d{2})?\s*(?:am|pm)?\b', ' __TIME__ ', text, flags=re.I)
    text = re.sub(r'-?\b\d+\.?\d*\b', ' __NUMBER__ ', text)
    text = re.sub(r'@\w+', ' __USER__ ', text)
    text = re.sub(r'#(\w+)', r' __HASHTAG__ \1 ', text)

    text = text.lower()
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')

    text = re.sub(r'[^\w\s]', '', text)

    words = word_tokenize(text)

    stop_words = set(stopwords.words('english'))
    words = [w for w in words if w not in stop_words or w.startswith('__')]

    # stemmer = PorterStemmer()
    # words = [w if w.startswith('__') and w.endswith('__') else stemmer.stem(w) for w in words]

    lemmatizer = WordNetLemmatizer()
    words = [w if w.startswith('__') and w.endswith('__') else lemmatizer.lemmatize(w) for w in words]

    text = ' '.join(words)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [15]:
clean_documents = documents.copy()
clean_documents['context'] = clean_documents['context'].progress_apply(text_normalize)
clean_documents.head()

100%|██████████| 18891/18891 [00:09<00:00, 1947.30it/s]


,context,c_id
0,architecturally school catholic character atop...,0
1,university notre dame student run number news ...,1
2,university major seat congregation holy cross ...,2
3,college engineering established __number__ how...,3
4,notre dame undergraduate student part one five...,4


## TF-IDF

In [16]:
vectorizer = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 3),
    analyzer='word'
)

vectorizer.fit(clean_documents['context'])

,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'word'
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(1, ...)"


In [17]:
X_test = vectorizer.transform(df_test_doc['context'].progress_apply(text_normalize))
Y_test = vectorizer.transform(df_test_questions['question'].progress_apply(text_normalize))

100%|██████████| 37782/37782 [00:03<00:00, 9582.40it/s]


In [18]:
similarity = (X_test @ Y_test.T).toarray()

In [19]:
pred_ids = []
for q_idx, question in tqdm(enumerate(df_test_questions.question), total=df_test_questions.shape[0]):
    best_idx = np.argmax(similarity[:, q_idx]).item()

    pred_ids.append(best_idx)


100%|██████████| 37782/37782 [00:01<00:00, 27081.46it/s]


In [20]:
submission = pd.DataFrame({'id': df_test_questions.index.tolist(), "pred_id": pred_ids})
submission.to_csv("sample_submission.csv", index=False)


## Embeding model

In [21]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

### bge small transformer

In [22]:
tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-small-en-v1.5')
model = AutoModel.from_pretrained('BAAI/bge-small-en-v1.5')
model.eval()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [23]:
docs = df_test_doc['context'].to_list()
ques = df_test_questions['question'].to_list()

# docs = df_test_doc['context'].progress_apply(text_normalize).to_list()
# ques = df_test_questions['question'].progress_apply(text_normalize).to_list()

In [24]:
len(docs)

18891

In [ ]:
docs_embs = []
model = model.to(device)

for doc in tqdm(docs):
    # Tokenize sentences
    encoded_input = tokenizer([doc], padding=True, truncation=True, return_tensors='pt')
    # for s2p(short query to long passage) retrieval task, add an instruction to query (not add instruction for passages)
    # encoded_input = tokenizer([instruction + q for q in queries], padding=True, truncation=True, return_tensors='pt')

    # Compute token embeddings
    with torch.no_grad():
        encoded_input = encoded_input.to(device)
        model_output = model(**encoded_input)
        # Perform pooling. In this case, cls pooling.
        sentence_embeddings = model_output[0][:, 0]

    # normalize embeddings
    sentence_embeddings = torch.flatten(F.normalize(sentence_embeddings, p=2, dim=1).cpu()[0])

    docs_embs.append(sentence_embeddings.numpy())
    
model = model.cpu()

docs_embs = np.array(docs_embs)

  0%|          | 0/18891 [00:00<?, ?it/s]

100%|██████████| 18891/18891 [00:49<00:00, 385.03it/s]


In [25]:
q_embs = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

for q in tqdm(ques):
    # Tokenize sentences
    encoded_input = tokenizer([q], padding=True, truncation=True, return_tensors='pt')
    # for s2p(short query to long passage) retrieval task, add an instruction to query (not add instruction for passages)
    # encoded_input = tokenizer([instruction + q for q in queries], padding=True, truncation=True, return_tensors='pt')

    # Compute token embeddings
    with torch.no_grad():
        encoded_input = encoded_input.to(device)
        model_output = model(**encoded_input)
        # Perform pooling. In this case, cls pooling.
        sentence_embeddings = model_output[0][:, 0]

    # normalize embeddings
    sentence_embeddings = torch.flatten(F.normalize(sentence_embeddings, p=2, dim=1).cpu()[0])

    q_embs.append(sentence_embeddings.numpy())
    
model = model.cpu()

q_embs = np.array(q_embs)

  0%|          | 86/37782 [00:00<01:28, 426.51it/s]

100%|██████████| 37782/37782 [01:26<00:00, 435.22it/s]


In [26]:
X_test = docs_embs
Y_test = q_embs

In [27]:
similarity = X_test @ Y_test.T

In [28]:
pred_ids = []
for q_idx, question in tqdm(enumerate(df_test_questions.question), total=df_test_questions.shape[0]):
    best_idx = np.argmax(similarity[:, q_idx]).item()

    pred_ids.append(best_idx)


100%|██████████| 37782/37782 [00:00<00:00, 38762.65it/s]


In [29]:
submission = pd.DataFrame({'id': df_test_questions.index.tolist(), "pred_id": pred_ids})
submission.to_csv("emb_subm_mynorm.csv", index=False)


### bge ranker

In [23]:
tokenizer = AutoTokenizer.from_pretrained('BAAI/bge-reranker-large')
model = AutoModelForSequenceClassification.from_pretrained('BAAI/bge-reranker-large')
model.eval()

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


XLMRobertaForSequenceClassification(
  (classifier): XLMRobertaClassificationHead(
    (dense): Linear(in_features=1024, out_features=1024, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (out_proj): Linear(in_features=1024, out_features=1, bias=True)
  )
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              

In [ ]:
model.eval()

pairs = [['what is panda?', 'hi']] #, ['what is panda?', 'The giant panda (Ailuropoda melanoleuca), sometimes called a panda bear or simply panda, is a bear species endemic to China.']]
with torch.no_grad():
    inputs = tokenizer(pairs, padding=True, truncation=True, return_tensors='pt', max_length=512)
    scores = model(**inputs, return_dict=True).logits.view(-1, ).float()
    print(scores)


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tensor([-5.6085])


In [ ]:
model.eval()

model = model.to(device)
pred_ids = []

for quesion in tqdm(ques):
    pairs = [[question, ans] for ans in docs]

    scores = []
    for i in range(0, len(pairs), 100):
        pair = pairs[i:min(i+100, len(pairs))]
        # print(pair)
        with torch.no_grad():
            inputs = tokenizer(pair, padding=True, truncation=True, return_tensors='pt', max_length=512).to(device)
            score = list(model(**inputs, return_dict=True).logits.view(-1, ).float().cpu().numpy())
            # print(score)
            scores = scores + score
    
    scores = np.array(scores)
    best_idx = np.argmax(similarity[:, q_idx]).item()
    pred_ids.append(best_idx)

model = model.cpu()
pred_ids


  0%|          | 0/37782 [00:00<?, ?it/s]


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.95 GiB. GPU 0 has a total capacity of 11.59 GiB of which 1.42 GiB is free. Including non-PyTorch memory, this process has 10.15 GiB memory in use. Of the allocated memory 9.92 GiB is allocated by PyTorch, and 13.33 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
submission = pd.DataFrame({'id': df_test_questions.index.tolist(), "pred_id": pred_ids})
submission.to_csv("ranker_subm.csv", index=False)
